# 01. Does the heart work?This notebook is the readable companion to `tests/test_validation.py`, which is theauthority. Nothing here asserts anything the test suite does not; the point is to *look*at the beat rather than to trust a green tick.Run `make validate` first, or just execute this notebook, which computes the table itself.

In [ ]:
import matplotlib.pyplot as pltimport numpy as npimport pandas as pdfrom hcmtwin import (HCM_GEOMETRY, HCM_MATERIAL, HEALTHY_GEOMETRY, HEALTHY_MATERIAL,                     RESTING_LOADING, observe, simulate)from hcmtwin.validation import validation_rowsfrom hcmtwin.viz import stylestyle.apply()pd.set_option("display.width", 200)pd.set_option("display.max_colwidth", 70)

## The gates

In [ ]:
table = validation_rows()print(f"{int(table['pass'].sum())} of {len(table)} gates pass")failures = table[~table["pass"]]display(failures if len(failures) else table.head(12))

## The beat itselfThree loops from the same equations. Only myosin availability, passive stiffness and wallvolume differ between them; nothing writes down an ejection fraction.

In [ ]:
runs = {    "healthy": (HEALTHY_GEOMETRY, HEALTHY_MATERIAL, 0.0),    "HCM, untreated": (HCM_GEOMETRY, HCM_MATERIAL, 0.0),    "HCM, 10 mg/day": (HCM_GEOMETRY, HCM_MATERIAL, 10.0),}traces = {}figure, ax = plt.subplots(figsize=(5.6, 4.4))for (label, (geom, mat, dose)), colour in zip(runs.items(), style.SERIES):    result = simulate(geom, mat, RESTING_LOADING, dose, record_trace=True)    traces[label] = (result, geom)    t = result.trace    v = np.append(t.cavity_volume_ml, t.cavity_volume_ml[0])    p = np.append(t.lv_pressure_mmhg, t.lv_pressure_mmhg[0])    ax.plot(v, p, color=colour, label=label)    ax.annotate(label, (v[int(np.argmax(p))], p.max()), xytext=(6, 5),                textcoords="offset points", color=colour, fontsize=9, fontweight="bold")ax.set_xlabel("Left-ventricular volume (mL)")ax.set_ylabel("Left-ventricular pressure (mmHg)")ax.set_xlim(left=0); ax.set_ylim(bottom=0)ax.set_title("Pressure-volume loops")plt.show()

## What each patient reports

In [ ]:
rows = []for label, (result, geom) in traces.items():    o = observe(result, geom, RESTING_LOADING)    rows.append({        "patient": label,        "EF": round(o.ejection_fraction, 3),        "EDV (mL)": round(o.edv_ml, 1),        "ESV (mL)": round(o.esv_ml, 1),        "SV (mL)": round(o.stroke_volume_ml, 1),        "EDP (mmHg)": round(o.end_diastolic_pressure_mmhg, 1),        "gradient (mmHg)": round(o.peak_lvot_gradient_mmhg, 1),        "wall (cm)": round(o.wall_thickness_cm, 2),        "E/e'": round(o.e_over_e_prime, 1),        "strain": round(o.peak_strain_amplitude, 3),        "ATP/work": round(o.atp_cost_per_stroke_work, 0),    })display(pd.DataFrame(rows))

### The finding worth pausing onGive the HCM **geometry** healthy **material** and the ejection fraction is *higher* thanthe diseased reference. Wall thickening alone raises ejection fraction, which is why areassuring ejection fraction in a thick-walled ventricle is close to uninformative. Whatthe wall alone does not produce is the elevated filling pressure or the energy cost.

In [ ]:
benign = observe(simulate(HCM_GEOMETRY, HEALTHY_MATERIAL, RESTING_LOADING, 0.0),                 HCM_GEOMETRY, RESTING_LOADING)diseased = observe(simulate(HCM_GEOMETRY, HCM_MATERIAL, RESTING_LOADING, 0.0),                   HCM_GEOMETRY, RESTING_LOADING)comparison = pd.DataFrame({    "thick wall, healthy tissue": [benign.ejection_fraction,                                   benign.end_diastolic_pressure_mmhg,                                   benign.e_over_e_prime,                                   benign.atp_cost_per_stroke_work],    "thick wall, HCM tissue": [diseased.ejection_fraction,                               diseased.end_diastolic_pressure_mmhg,                               diseased.e_over_e_prime,                               diseased.atp_cost_per_stroke_work],}, index=["ejection fraction", "filling pressure (mmHg)", "E/e' surrogate", "ATP per unit work"])display(comparison.round(2))

## Within-beat waveformsWorth a look because the pressure-volume loop hides the timing. Note that the myosinpopulations move within the beat: the parked pool is drawn down during ejection byforce-dependent recruitment and refills in diastole.

In [ ]:
result, _ = traces["HCM, untreated"]t = result.tracefigure, axes = plt.subplots(2, 2, figsize=(9.5, 5.6), sharex=True)axes[0, 0].plot(t.time_s, t.lv_pressure_mmhg, color=style.SERIES[0], label="LV")axes[0, 0].plot(t.time_s, t.arterial_pressure_mmhg, color=style.SERIES[1], label="aortic")axes[0, 0].set_ylabel("Pressure (mmHg)"); axes[0, 0].legend(); axes[0, 0].set_title("Pressures")axes[0, 1].plot(t.time_s, t.aortic_flow_ml_per_s, color=style.SERIES[0], label="aortic")axes[0, 1].plot(t.time_s, t.mitral_flow_ml_per_s, color=style.SERIES[1], label="mitral")axes[0, 1].set_ylabel("Flow (mL/s)"); axes[0, 1].legend(); axes[0, 1].set_title("Valve flows")axes[1, 0].plot(t.time_s, t.attached, color=style.SERIES[0], label="attached")axes[1, 0].plot(t.time_s, t.parked, color=style.SERIES[1], label="parked")axes[1, 0].plot(t.time_s, t.available, color=style.SERIES[2], label="available")axes[1, 0].set_ylabel("Fraction of heads"); axes[1, 0].set_xlabel("Time (s)")axes[1, 0].legend(); axes[1, 0].set_title("Myosin populations")axes[1, 1].plot(t.time_s, t.calcium_um, color=style.SERIES[0])axes[1, 1].set_ylabel("Calcium (uM)"); axes[1, 1].set_xlabel("Time (s)")axes[1, 1].set_title("Prescribed calcium transient")plt.tight_layout(); plt.show()print("S + D + A departs from 1 by at most "      f"{float(result.summary.population_error):.2e}")